In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym

# ---------- Setup ----------
SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=False
)

N_STATES, N_ACTIONS = 16, 4
TERMINAL = {5, 7, 11, 12, 15}


def one_hot(s):
    v = np.zeros(N_STATES, dtype=np.float32)
    v[s] = 1.0
    return v


# ---------- 1. Collect imperfect dataset ----------
def behavior_policy():
    if np.random.rand() < 0.3:
        return env.action_space.sample()
    return np.random.choice([1, 2])  # Down / Right


S, A, R, S2, D = [], [], [], [], []

for _ in range(300):
    s, _ = env.reset()
    done = False

    while not done:
        a = behavior_policy()
        s2, r, term, trunc, _ = env.step(a)
        done = term or trunc

        S.append(s)
        A.append(a)
        R.append(r)
        S2.append(s2)
        D.append(float(done))

        s = s2

S = np.array(S)
A = np.array(A)
S2 = np.array(S2)
R = np.array(R, dtype=np.float32)
D = np.array(D, dtype=np.float32)


# Split state-action pairs into common vs rare
pair_counts = np.zeros(
    (N_STATES, N_ACTIONS), dtype=int
)

for s, a in zip(S, A):
    pair_counts[s, a] += 1

valid = [
    s for s in range(N_STATES)
    if s not in TERMINAL
]

common = [
    (s, a)
    for s in valid
    for a in range(4)
    if pair_counts[s, a] >= 5
]

rare = [
    (s, a)
    for s in valid
    for a in range(4)
    if pair_counts[s, a] < 5
]


# ---------- 2. Q-network ----------
class QNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(N_STATES, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, N_ACTIONS)
        )

    def forward(self, x):
        return self.net(x)


# ---------- 3. Bellman + CQL loss ----------
def cql_loss(
    q_net, target_net, s, a, r, s2, done,
    gamma, alpha
):
    q_all = q_net(s)

    q_data = q_all.gather(
        1, a.unsqueeze(1)
    ).squeeze(1)

    with torch.no_grad():
        target = r + gamma * (1 - done) * (
            target_net(s2).max(dim=1).values
        )

    bellman = F.mse_loss(q_data, target)

    cql_pen = (
        torch.logsumexp(q_all, dim=1) - q_data
    ).mean()

    return bellman + alpha * cql_pen


# ---------- 4. Offline training ----------
S_T = torch.tensor(
    np.array([one_hot(x) for x in S])
)

S2_T = torch.tensor(
    np.array([one_hot(x) for x in S2])
)

A_T = torch.tensor(A, dtype=torch.long)
R_T = torch.tensor(R)
D_T = torch.tensor(D)


def train(alpha, steps=3000, batch=64,
          lr=1e-3, gamma=0.99):

    torch.manual_seed(SEED)
    np.random.seed(SEED)

    q = QNet()
    tgt = QNet()
    tgt.load_state_dict(q.state_dict())

    opt = torch.optim.Adam(q.parameters(), lr=lr)

    for step in range(steps):
        i = np.random.randint(
            0, len(S), batch
        )

        loss = cql_loss(
            q, tgt,
            S_T[i], A_T[i], R_T[i],
            S2_T[i], D_T[i],
            gamma, alpha
        )

        opt.zero_grad()
        loss.backward()
        opt.step()

        if step % 100 == 0:
            tgt.load_state_dict(q.state_dict())

    return q


# ---------- 5. Helpers ----------
def q_table(q):
    with torch.no_grad():
        states = torch.tensor(
            np.array([
                one_hot(s) for s in range(N_STATES)
            ])
        )
        return q(states).numpy()


def evaluate(q):
    s, _ = env.reset()

    total = 0.0
    done = False
    steps = 0

    while not done and steps < 50:
        with torch.no_grad():
            state_tensor = torch.tensor(
                one_hot(s)
            ).unsqueeze(0)

            a = q(state_tensor).argmax().item()

        s, r, term, trunc, _ = env.step(a)
        done = term or trunc

        total += r
        steps += 1

    return total


def avg_q(qt, pairs):
    if not pairs:
        return 0.0

    return float(
        np.mean([qt[s, a] for s, a in pairs])
    )


# ---------- 6. Run and print key results ----------
dqn_model = train(alpha=0.0)
cql_model = train(alpha=1.0)

dqn_q = q_table(dqn_model)
cql_q = q_table(cql_model)

dqn_ret = evaluate(dqn_model)
cql_ret = evaluate(cql_model)

print(
    f"Dataset: {len(S)} transitions | "
    f"Left used {100*np.mean(A == 0):.0f}%, "
    f"Down {100*np.mean(A == 1):.0f}%, "
    f"Right {100*np.mean(A == 2):.0f}%, "
    f"Up {100*np.mean(A == 3):.0f}%"
)

print(
    f"Rare/unseen (state,action) pairs: "
    f"{len(rare)} of {len(rare) + len(common)}\n"
)

print(
    f"{'':<12}{'Return':>8}"
    f"{'Avg Q common':>15}"
    f"{'Avg Q rare':>13}"
    f"{'Rare Q > 1.0':>15}"
)

for name, qt, ret in [
    ("Offline DQN", dqn_q, dqn_ret),
    ("CQL", cql_q, cql_ret)
]:
    over = sum(
        qt[s, a] > 1.0 for s, a in rare
    )

    print(
        f"{name:<12}{ret:>8.1f}"
        f"{avg_q(qt, common):>15.3f}"
        f"{avg_q(qt, rare):>13.3f}"
        f"{over:>11d}/{len(rare)}"
    )

print(
    "\nNote: the max possible reward is 1.0, "
    "so any Q above 1.0 is overestimation."
)

env.close()

Dataset: 1282 transitions | Left used 7%, Down 42%, Right 43%, Up 8%
Rare/unseen (state,action) pairs: 6 of 44

              Return   Avg Q common   Avg Q rare   Rare Q > 1.0
Offline DQN      0.0          0.824        0.488          0/6
CQL              0.0          0.954        0.158          0/6

Note: the max possible reward is 1.0, so any Q above 1.0 is overestimation.
